<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Vector concepts: `cutlass.Vector` registers vs memory

So far data has lived in `cutlass.Array`s. But an `Array` is only a *handle to memory* -- a pointer
into global, shared, or local space. The arithmetic itself happens in **registers**, and that is
what `cutlass.Vector` is: an **immutable value** held in a thread's registers, with no address of
its own.

Load and store connect the two:

- **load** -- slicing an Array pulls elements into a Vector: `arr[base:V]` reads `V` contiguous
  elements (registers <- memory).
- **store** -- assigning a Vector into a slice writes them back: `arr[base:V] = vec`
  (memory <- registers).

What you hold decides what you can do. An `Array` you index and assign in place; a `Vector` you can
only transform into a *new* `Vector`.

**You'll learn:** the **Array (memory) vs Vector (registers)** split; how the slice `arr[base:V]`
(`V` is a *count*) loads a `cutlass.Vector`; element-wise SIMD arithmetic with **scalar broadcast**
(`vec * A + B`); the vectorized select `cutlass.vector.where`; a constant vector from
`cutlass.vector.full`; and the in-register **horizontal reduction** `vec.reduce("add")` that
collapses all lanes to one scalar -- no shared memory, no shuffles.

**Runs on:** any CUDA GPU. **Prereq:** the `01_array_concepts` notebook (same `arr[base:V]` slice).

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. Array is memory, Vector is registers

| type | what it is | operations |
|---|---|---|
| `cutlass.Array` | a handle to memory (global / shared / local) | `arr[i]` scalar load/store; `arr[i:V]` vector load -> `Vector`; `arr[i:V] = vec` vector store (registers -> memory) |
| `cutlass.Vector` | an immutable value in a thread's registers | `vec * A + B` element-wise + broadcast; `cutlass.vector.where(c, a, b)` vectorized select; `vec.reduce("add")` horizontal fold -> scalar |

A `Vector` is a **value, not storage**: there is no element assignment (`vec[i] = x` raises
`TypeError`). To "change" a Vector you apply an operation that produces a *new* one -- exactly like
adding to a Python `int`.

The kernel below runs the whole round-trip. Each thread loads one `V`-wide tile into a Vector,
computes `act(A*x + B)` entirely in registers -- where `act` is picked from an `OPS` **dict** at
trace time -- stores the result, and also folds the tile to a single scalar with `reduce`.

In [ ]:
@cute.kernel
def vector_concepts_kernel(
    x: cutlass.Array, y: cutlass.Array, s: cutlass.Array,
    N: cutlass.Int32, A: cutlass.Constexpr, B: cutlass.Constexpr, V: cutlass.Constexpr,
    act: cutlass.Constexpr = "relu",
):
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    bdx, _, _ = cute.arch.block_dim()
    tidx = bx * bdx + tx
    base = tidx * V                      # this thread owns the V-wide tile starting here

    if base < N:
        # Step 1. Load: one V-wide vector load pulls the tile into registers.
        # A cute.printf after each Vector op makes the per-stage transform visible (tile 0 only).
        vec = x[base:V]                  # memory -> registers
        if tidx == 0:
            cute.printf(f"load    x[base:V]  = [{vec[0]} {vec[1]} {vec[2]} {vec[3]}]")

        # Step 2. Affine map: scalars A, B broadcast across all V lanes (SIMD).
        affine = vec * A + B
        if tidx == 0:
            cute.printf(f"affine  vec*A + B  = [{affine[0]} {affine[1]} {affine[2]} {affine[3]}]")

        # Step 3. Activation. OPS maps a name to a vector op; `act` is a Constexpr, so
        # OPS[act] picks the op at trace time and the kernel bakes in only that one.
        zeros = cutlass.vector.full((V,), 0.0, dtype=cutlass.Float32)
        OPS = {
            "relu":   lambda u: cutlass.vector.where(u > zeros, u, zeros),
            "square": lambda u: u * u,
            "abs":    lambda u: cutlass.vector.where(u > zeros, u, zeros - u),
        }
        activated = OPS[act](affine)
        if tidx == 0:
            cute.printf(f"act ({act}) = [{activated[0]} {activated[1]} {activated[2]} {activated[3]}]")

        # Step 4. Horizontal fold: V lanes -> one scalar, in registers.
        total = activated.reduce("add")
        if tidx == 0:
            cute.printf(f"reduce  sum        = {total}")

        # Step 5. Store: one V-wide vector store writes the tile back, plus the per-tile sum.
        y[base:V] = activated            # registers -> memory
        s[tidx] = total

## 2. Launch: one V-wide tile per thread

The grid covers all `N // V` tiles with 128-thread blocks. `A`, `B`, and `V` are `Constexpr`, so the
coefficients fold into constants and the load / store / reduce become fixed-width instructions.

In [ ]:
@cute.jit
def vector_concepts(
    x: cutlass.Array, y: cutlass.Array, s: cutlass.Array,
    N: cutlass.Int32, A: cutlass.Constexpr, B: cutlass.Constexpr, V: cutlass.Constexpr,
    act: cutlass.Constexpr = "relu",
):
    block = (128, 1, 1)
    tiles = N // V                                       # one thread per V-wide tile
    grid = ((tiles + block[0] - 1) // block[0], 1, 1)
    vector_concepts_kernel(x, y, s, N, A, B, V, act).launch(grid=grid, block=block)

## 3. Run and check against PyTorch

`cutlass.Array` parameters take PyTorch CUDA tensors directly via `cute.runtime.from_dlpack`
(zero-copy). We run the same kernel for each activation in the `OPS` dict and check both outputs
against the matching PyTorch op: the element-wise tile, and the per-tile row-sum.

In [ ]:
N, V = 1 << 20, 4          # 1,048,576 elements, one 4-wide tile per thread
A, B = 2.0, 1.0            # y = act(A*x + B)

x = torch.randn(N, dtype=torch.float32, device="cuda")

# The SAME kernel, specialized to each activation in OPS by name. `act` is a Constexpr,
# so each launch traces just one op out of the dict -- a compile-time dispatch table.
refs = {"relu": torch.relu, "square": lambda t: t * t, "abs": torch.abs}
for act in ("relu", "square", "abs"):
    y = torch.zeros(N, dtype=torch.float32, device="cuda")
    s = torch.zeros(N // V, dtype=torch.float32, device="cuda")   # one row-sum per tile
    vector_concepts(
        cute.runtime.from_dlpack(x), cute.runtime.from_dlpack(y), cute.runtime.from_dlpack(s),
        N, A, B, V, act,
    )
    ref = refs[act](A * x + B)
    torch.testing.assert_close(y.cpu(), ref.cpu(), atol=1e-2, rtol=1e-2)
    torch.testing.assert_close(s.cpu(), ref.reshape(N // V, V).sum(dim=-1).cpu(), atol=1e-2, rtol=1e-2)
    print(f"PASS  act={act}")

# Expected output (tile 0 prints once per activation; lane values depend on the random input):
# load    x[base:V]  = [...]
# affine  vec*A + B  = [...]
# act (relu) = [...]
# reduce  sum        = ...
# PASS  act=relu
# ... then act=square, act=abs ...

## Try it yourself

1. **A Vector is immutable.** Add `vec[0] = cutlass.Float32(9.0)` -- it raises `TypeError`, because a
   Vector is a value, not storage. You can only build a *new* Vector from it.
2. **Add an activation.** Add `"gelu": lambda u: ...` to the `OPS` dict and a matching reference --
   the dict is a trace-time dispatch table, so a new entry is a new specialized kernel for free.
3. **Other reductions.** Swap `reduce("add")` for `reduce("max")` (and adjust the torch reference);
   `reduce("mul")` and `reduce("min")` work too.
4. **Build a Vector from scalars.** Replace the load with
   `vec = cutlass.Vector.from_elements((1.0, 2.0, 3.0, 4.0), cutlass.Float32)` -- a Vector need not
   come from memory.
5. **Convert types.** Insert `vec = vec.to(cutlass.Float16)` before the affine map (store into an
   `fp16` output and widen the tolerance). `Vector.to(...)` converts element types; `.bitcast(...)`
   reinterprets the same bits as another type.